# Clase 3: Simulación Avanzada de Drift

## Objetivos

1. Entender tipos de drift: Covariate, Label, Feature
2. Simular drift artificial
3. Analizar impacto en métricas
4. Usar Evidently para detección
5. Comparar escenarios en MLflow

---

## Tipos de Drift

### 1. Covariate Shift
- **Qué cambia**: Distribución de features (X)
- **Relación Y|X**: Se mantiene igual
- **Ejemplo**: Nuevos usuarios con diferente perfil
- **Solución**: Re-weight o re-train

### 2. Label Shift
- **Qué cambia**: Proporción de clases (P(Y))
- **Ejemplo**: época de alta demanda vs baja demanda
- **Impacto**: Afecta métricas como Recall
- **Solución**: Ajustar umbrales de decisión

### 3. Feature Drift
- **Qué cambia**: Calidad o rango de features
- **Ejemplo**: Outliers, valores faltantes, ruido
- **Impacto**: Degradación directa del modelo
- **Solución**: Data cleaning, feature engineering

---


In [ ]:
# PASO 1: Setup
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("\u2713 Librerías importadas")

# Cargar datos
df = pd.read_csv('/app/data/dataset.csv')
X = df.drop('target', axis=1).values
y = df['target'].values
feature_names = df.drop('target', axis=1).columns.tolist()

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Entrenar modelo base
model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model.fit(X_train, y_train)

print("\u2713 Dataset cargado")
print(f"\u2713 Modelo entrenado en {len(X_train)} muestras")

## Escenario 1: SIN DRIFT (Baseline)

Datos nuevos con distribución similar a los datos de entrenamiento.

In [ ]:
# ESCENARIO 1: SIN DRIFT
print("=" * 60)
print("ESCENARIO 1: SIN DRIFT (BASELINE)")
print("=" * 60)

# Generar datos sin drift (similar a test)
indices = np.random.choice(len(X_test), size=100, replace=True)
X_baseline = X_test[indices] + np.random.normal(0, 0.01, X_test[indices].shape)
y_baseline = y_test[indices]

# Predicciones
y_pred_baseline = model.predict(X_baseline)

# Métricas
acc_baseline = accuracy_score(y_baseline, y_pred_baseline)
f1_baseline = f1_score(y_baseline, y_pred_baseline)

print(f"Accuracy: {acc_baseline:.4f}")
print(f"F1-Score: {f1_baseline:.4f}")
print("\nCaracterísticas: Distribución normal sin cambios")

## Escenario 2: COVARIATE SHIFT

Features cambian, pero la relación Y|X se mantiene.

In [ ]:
# ESCENARIO 2: COVARIATE SHIFT
print("\n" + "=" * 60)
print("ESCENARIO 2: COVARIATE SHIFT")
print("=" * 60)

# Aplicar shift a features
indices = np.random.choice(len(X_test), size=100, replace=True)
X_covariate = X_test[indices].copy().astype(float)

# Shift: aumentar media de features
for i in range(X_covariate.shape[1]):
    X_covariate[:, i] = X_covariate[:, i] + np.random.normal(0.5, 0.2, 100)

y_covariate = y_test[indices]
y_pred_covariate = model.predict(X_covariate)

acc_covariate = accuracy_score(y_covariate, y_pred_covariate)
f1_covariate = f1_score(y_covariate, y_pred_covariate)

print(f"Accuracy: {acc_covariate:.4f} (cambio: {acc_covariate - acc_baseline:+.4f})")
print(f"F1-Score: {f1_covariate:.4f} (cambio: {f1_covariate - f1_baseline:+.4f})")
print("\nCaracterísticas: Media de features +0.5")

## Escenario 3: LABEL SHIFT

Proporción de clases cambia, features igual.

In [ ]:
# ESCENARIO 3: LABEL SHIFT
print("\n" + "=" * 60)
print("ESCENARIO 3: LABEL SHIFT")
print("=" * 60)

# Cambiar distribución de clases (80% clase 0, 20% clase 1)
n_samples = 100
class_0_count = int(0.8 * n_samples)
class_1_count = n_samples - class_0_count

indices_0 = np.random.choice(np.where(y_test == 0)[0], size=class_0_count, replace=True)
indices_1 = np.random.choice(np.where(y_test == 1)[0], size=class_1_count, replace=True)

X_label = np.vstack([X_test[indices_0], X_test[indices_1]])
y_label = np.concatenate([y_test[indices_0], y_test[indices_1]])

y_pred_label = model.predict(X_label)

acc_label = accuracy_score(y_label, y_pred_label)
f1_label = f1_score(y_label, y_pred_label)

print(f"Accuracy: {acc_label:.4f} (cambio: {acc_label - acc_baseline:+.4f})")
print(f"F1-Score: {f1_label:.4f} (cambio: {f1_label - f1_baseline:+.4f})")
print(f"\nDistribución de clases:")
print(f"  - Clase 0: {(y_label == 0).sum()} muestras")
print(f"  - Clase 1: {(y_label == 1).sum()} muestras")

## Escenario 4: FEATURE DRIFT

Outliers, ruido y cambios de escala.

In [ ]:
# ESCENARIO 4: FEATURE DRIFT
print("\n" + "=" * 60)
print("ESCENARIO 4: FEATURE DRIFT")
print("=" * 60)

# Generar datos con drift
indices = np.random.choice(len(X_test), size=100, replace=True)
X_feature = X_test[indices].copy().astype(float)

# Agregar outliers (10% de datos)
outlier_indices = np.random.choice(100, size=10, replace=False)
for idx in outlier_indices:
    feat_idx = np.random.randint(0, X_feature.shape[1])
    X_feature[idx, feat_idx] = np.random.uniform(-3, 3)

# Ruido gaussiano
X_feature = X_feature + np.random.normal(0, 0.3, X_feature.shape)

y_feature = y_test[indices]
y_pred_feature = model.predict(X_feature)

acc_feature = accuracy_score(y_feature, y_pred_feature)
f1_feature = f1_score(y_feature, y_pred_feature)

print(f"Accuracy: {acc_feature:.4f} (cambio: {acc_feature - acc_baseline:+.4f})")
print(f"F1-Score: {f1_feature:.4f} (cambio: {f1_feature - f1_baseline:+.4f})")
print("\nCaracterísticas: 10% outliers + ruido gaussiano")

## Análisis Comparativo

In [ ]:
# COMPARACIÓN DE ESCENARIOS
print("\n" + "=" * 60)
print("COMPARACIÓN DE ESCENARIOS")
print("=" * 60)

comparison = pd.DataFrame({
    'Escenario': ['Sin Drift', 'Covariate Shift', 'Label Shift', 'Feature Drift'],
    'Accuracy': [acc_baseline, acc_covariate, acc_label, acc_feature],
    'F1-Score': [f1_baseline, f1_covariate, f1_label, f1_feature],
    'Degradación Acc': [0, acc_covariate - acc_baseline, acc_label - acc_baseline, acc_feature - acc_baseline]
})

print(comparison.to_string(index=False))

# Escenario más crítico
degradations = {
    'Covariate': acc_covariate - acc_baseline,
    'Label': acc_label - acc_baseline,
    'Feature': acc_feature - acc_baseline
}
worst = min(degradations.items(), key=lambda x: x[1])

print(f"\n⚠️  ESCENARIO MÁS CRÍTICO: {worst[0]} Shift")
print(f"   Degradación: {worst[1]:.4f} en accuracy")

## Conclusiones

1. **Todos los tipos de drift afectan el modelo**
2. **Feature Drift puede ser el más crítico** (outliers, ruido)
3. **Label Shift afecta proporcionalmente las métricas**
4. **Monitoreo es esencial** para detectar estos cambios
5. **Re-entrenamiento** puede ser necesario cuando degradación > umbral

### Recomendaciones

- Monitorear **Daily/Weekly** en producción
- Establecer **thresholds de alerta**
- Mantener **logs de drift detection**
- Implementar **automated re-training** si aplica
- Comunicar **degradación** a stakeholders